# UdaPlay — Building an AI Game Research Agent

This notebook guides you step-by-step through completing the **UdaPlay** starter project into a fully working AI agent.

**What you will build:** An agent that answers questions about the video game industry by first querying a local vector database (RAG), evaluating whether the results are good enough, and falling back to a live web search when they are not.

**Prerequisites:**
- Python 3.11+
- This notebook must be opened from inside the `building_agents_start/starter/` directory (so that `lib/` imports resolve correctly).
- A `.env` file in the same directory with the following keys:

```
OPENAI_API_KEY=sk-...
CHROMA_OPENAI_API_KEY=sk-...   # can be the same as OPENAI_API_KEY
TAVILY_API_KEY=tvly-...
```

---

### Project architecture

```
starter/
├── lib/                    ← pre-built helper library
├── games/                  ← 15 game JSON files
├── chromadb/               ← persistent vector store (created on first run)
├── .env                    ← API keys
└── building_agent_final.ipynb   ← this file
```

### What you will implement

| Step | What | Why |
|------|------|-----|
| 0 | Install dependencies | baseline environment |
| 1 | Fix 3 bugs in `lib/` | the library ships with intentional bugs |
| 2 | Vector DB setup | load 15 games into ChromaDB |
| 3 | `EvaluationReport` Pydantic model | structured LLM output |
| 4 | 3 tools (`retrieve_game`, `evaluate_retrieval`, `game_web_search`) | agent capabilities |
| 5 | Agent + system prompt | orchestration |
| 6 | Test queries | validation |

---

## Step 0 — Install dependencies

In [ ]:
!pip install chromadb>=1.0.4 openai>=1.73.0 pydantic>=2.11.3 python-dotenv>=1.1.0 tavily-python>=0.5.4

---

## Step 1 — Fix 3 bugs in `lib/`

The `lib/` directory ships with three intentional bugs. You must fix them before running the rest of this notebook. Open each file in your editor, apply the change described below, and save. After all three fixes are in place, run the **verification cell** at the end of this section.

> **Why fix these before anything else?** Running the agent without these fixes will produce confusing errors deep inside the library.

### Fix 1 — `lib/llm.py` line 24: missing `base_url` in the fallback branch

**Problem:** When no `api_key` is supplied the code falls through to a plain `OpenAI()` constructor that hits the default OpenAI endpoint instead of the configured one, causing a 401 error.

Find line 24 and change it from:

```python
self.client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=api_key) if api_key else OpenAI()
```

to:

```python
self.client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=api_key) if api_key else OpenAI(base_url="https://openai.vocareum.com/v1")
```

### Fix 2 — `lib/vector_db.py`: remove `'distances'` from the `get()` `include` list

**Problem:** ChromaDB's `get()` method does not accept `'distances'` in its `include` list — that field is only valid for `query()`. Passing it raises a `ValueError`.

Find the `get` method (around line 138). Change:

```python
return self._collection.get(
    ids=ids,
    where=where,
    limit=limit,
    include=['documents', 'distances', 'metadatas']
)
```

to:

```python
return self._collection.get(
    ids=ids,
    where=where,
    limit=limit,
    include=['documents', 'metadatas']
)
```

### Fix 3 — `lib/agents.py`: fill in missing optional parameters before calling a tool

**Problem:** When the LLM omits optional parameters from a tool call, calling `tool(**function_args)` raises a `TypeError`. The agent must fill in default values automatically.

Inside the `_tool_step` method, find the block that looks up and calls the tool (approximately line 100):

```python
tool = next((t for t in self.tools if t.name == function_name), None)
if tool:
    result = str(tool(**function_args))
    tool_message = ToolMessage(...)
```

Replace it with:

```python
tool = next((t for t in self.tools if t.name == function_name), None)
if tool:
    # Fill in default values for any optional parameters the LLM omitted
    import inspect
    sig = inspect.signature(tool.func)
    for param_name, param in sig.parameters.items():
        if param.default != inspect.Parameter.empty and param_name not in function_args:
            function_args[param_name] = param.default

    try:
        result = str(tool(**function_args))
    except TypeError as e:
        import inspect
        sig = inspect.signature(tool.func)
        for param_name, param in sig.parameters.items():
            if param_name not in function_args and param.default != inspect.Parameter.empty:
                function_args[param_name] = param.default
        result = str(tool(**function_args))
    tool_message = ToolMessage(...)
```

> Keep `tool_message = ToolMessage(...)` and everything that follows exactly as it was — only the lines above it change.

In [ ]:
# Verification — run this cell after applying all three fixes.
# Every line should print ✅. If you see ❌ go back and re-apply the corresponding fix.

import importlib
import lib.llm, lib.vector_db, lib.agents
importlib.reload(lib.llm)
importlib.reload(lib.vector_db)
importlib.reload(lib.agents)

import inspect
llm_src  = inspect.getsource(lib.llm.LLM.__init__)
vec_src  = inspect.getsource(lib.vector_db.VectorStore.get)
agt_src  = inspect.getsource(lib.agents.Agent._tool_step)

fix1_ok = 'else OpenAI(base_url' in llm_src
fix2_ok = "'distances'" not in vec_src
fix3_ok = 'inspect.signature(tool.func)' in agt_src

print(f'Fix 1 (lib/llm.py     — default base_url)  : {"✅" if fix1_ok else "❌ not applied yet"}')
print(f'Fix 2 (lib/vector_db.py — remove distances) : {"✅" if fix2_ok else "❌ not applied yet"}')
print(f'Fix 3 (lib/agents.py  — optional params)   : {"✅" if fix3_ok else "❌ not applied yet"}')

---

## Step 2 — Imports and environment setup

In [ ]:
import os
import json
import sys
import importlib.util
from typing import List, Dict, Any, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from openai import OpenAI

# Udacity / pysqlite3 compatibility shim (safe to leave in; no-op locally)
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

from lib.vector_db import VectorStore
from lib.agents import Agent
from lib.tooling import tool
from lib.llm import LLM
from lib.parsers import PydanticOutputParser
from lib.documents import Document, Corpus

load_dotenv()

OPENAI_API_KEY        = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY") or OPENAI_API_KEY
TAVILY_API_KEY        = os.getenv("TAVILY_API_KEY")

print(f'OPENAI_API_KEY  : {"✅ found" if OPENAI_API_KEY  else "❌ missing — check .env"}')
print(f'TAVILY_API_KEY  : {"✅ found" if TAVILY_API_KEY  else "❌ missing — check .env"}')

---

## Step 3 — Set up the Vector Database (Part 1)

This step:
1. Creates (or reuses) a persistent ChromaDB collection called `udaplay`.
2. Reads all 15 game JSON files from `games/`.
3. Embeds each document with `text-embedding-ada-002` and stores it.

The function is **idempotent** — if the collection already contains documents it returns immediately without re-indexing.

Each document is stored in this format:
```
[Platform] Name (Year) - Description
```

In [ ]:
# --- ChromaDB client + embedding function ---

SCRIPT_DIR       = os.getcwd()
CHROMA_DB_PATH   = os.path.join(SCRIPT_DIR, "chromadb")

chroma_client = chromadb.Client(Settings(persist_directory=CHROMA_DB_PATH))

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY
)

GAMES_COLLECTION_NAME = "udaplay"
games_store: Optional[VectorStore] = None

print(f'ChromaDB persistence path: {CHROMA_DB_PATH}')

In [ ]:
def setup_vector_database() -> VectorStore:
    """Initialise ChromaDB and load game documents (idempotent)."""
    global games_store

    collection = None

    # 1. Try to reuse an existing, healthy collection
    try:
        existing_collection = chroma_client.get_collection(name=GAMES_COLLECTION_NAME)
        temp_store = VectorStore(existing_collection)

        try:
            collection_id  = existing_collection.id
            test_embedding = [[0.0] * 1536]  # text-embedding-ada-002 produces 1536-dim vectors

            test_results = chroma_client._query(
                collection_id=collection_id,
                query_embeddings=test_embedding,
                n_results=1,
                include=['documents', 'metadatas']
            )

            documents = test_results.get('documents', [[]])
            if documents and len(documents) > 0 and len(documents[0]) > 0:
                all_data    = temp_store.get()
                total_count = len(all_data.get('ids', [])) if all_data.get('ids') else 0
                if total_count > 0:
                    print(f"Collection '{GAMES_COLLECTION_NAME}' already contains {total_count} documents — skipping ingestion.")
                    games_store = temp_store
                    return games_store

            collection  = existing_collection
            games_store = temp_store

        except Exception as e:
            print(f'Existing collection appears corrupt, recreating... ({e})')
            try:
                chroma_client.delete_collection(name=GAMES_COLLECTION_NAME)
            except Exception:
                pass
            collection = None

    except Exception:
        collection = None

    # 2. Create a new collection when none exists
    if collection is None:
        collection  = chroma_client.create_collection(name=GAMES_COLLECTION_NAME)
        games_store = VectorStore(collection)
        print(f"Created new collection '{GAMES_COLLECTION_NAME}'.")

    # 3. Load game JSON files
    games_dir = os.path.join(SCRIPT_DIR, 'games')
    if not os.path.exists(games_dir):
        raise FileNotFoundError(f'games/ directory not found at: {games_dir}')

    documents = []
    for file_name in sorted(os.listdir(games_dir)):
        if not file_name.endswith('.json'):
            continue
        file_path = os.path.join(games_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as f:
            game = json.load(f)

        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
        doc_id  = os.path.splitext(file_name)[0]
        documents.append(Document(id=doc_id, content=content, metadata=game))

    # 4. Compute embeddings manually (ChromaDB ≥1.0 internal-API workaround)
    if documents:
        contents  = [doc.content   for doc in documents]
        ids       = [doc.id        for doc in documents]
        metadatas = [doc.metadata  for doc in documents]

        openai_client = OpenAI(
            base_url='https://openai.vocareum.com/v1',
            api_key=CHROMA_OPENAI_API_KEY
        )
        embeddings_response = openai_client.embeddings.create(
            model='text-embedding-ada-002',
            input=contents
        )
        embeddings = [item.embedding for item in embeddings_response.data]

        chroma_client._add(
            collection_id=collection.id,
            documents=contents,
            ids=ids,
            metadatas=metadatas,
            embeddings=embeddings
        )
        print(f'Loaded {len(documents)} game documents into the vector store.')

    return games_store


print('setup_vector_database() defined')

In [ ]:
# Run the setup — safe to re-run; it will skip ingestion if the DB already exists
setup_vector_database()

In [ ]:
# Quick sanity check
all_data = games_store.get()
print(f'Total documents in DB : {len(all_data["ids"])}')
print(f'First entry           : {all_data["documents"][0]}')

---

## Step 4 — Evaluation Pydantic model (Part 2)

`EvaluationReport` is the structured output that the LLM-as-judge returns when assessing whether the retrieved documents are sufficient to answer a question.

- `useful` — `True` if the docs contain enough information, `False` if a web search is needed.
- `description` — a human-readable explanation of the decision.

The `Field(description=...)` values are included in the prompt sent to the LLM, which helps it understand what it should return.

In [ ]:
class EvaluationReport(BaseModel):
    """Structured evaluation report for retrieval quality."""
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question"
    )
    description: str = Field(
        description="A detailed explanation of the evaluation decision"
    )

print('EvaluationReport defined')
print(EvaluationReport.model_json_schema())

---

## Step 5 — Implement the three agent tools (Part 2)

The agent needs exactly three tools:

| Tool | Purpose |
|------|---------|
| `retrieve_game` | Semantic search inside ChromaDB |
| `evaluate_retrieval` | Ask an LLM judge whether the results are good enough |
| `game_web_search` | Fallback — live web search via Tavily |

The `@tool` decorator exposes each function to the agent. The **docstring** is what the LLM reads to decide when and how to call the tool — write it carefully.

### Tool 1 — `retrieve_game`

Embeds the query and searches the ChromaDB collection for the closest matching game documents. Returns a list of dicts — one per result — sorted by similarity score.

In [ ]:
@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """
    Semantic search: finds the most relevant game documents in the vector database.

    Args:
        query: A question about the video game industry.

    Returns:
        A list of game documents, each containing:
        - Platform: e.g. Game Boy, PlayStation 5, Xbox 360
        - Name: title of the game
        - YearOfRelease: release year for that platform
        - Description: additional details
        - Genre, Publisher, Content, similarity
    """
    global games_store, chroma_client

    try:
        if games_store is None:
            games_store = setup_vector_database()
        if games_store is None:
            return []

        openai_client = OpenAI(
            base_url='https://openai.vocareum.com/v1',
            api_key=CHROMA_OPENAI_API_KEY
        )
        query_embedding_response = openai_client.embeddings.create(
            model='text-embedding-ada-002',
            input=[query]
        )
        query_embeddings = [item.embedding for item in query_embedding_response.data]

        collection_id = games_store._collection.id
        results = chroma_client._query(
            collection_id=collection_id,
            query_embeddings=query_embeddings,
            n_results=10,
            include=['documents', 'distances', 'metadatas']
        )

        retrieved_games = []
        documents = results.get('documents', [[]])
        metadatas = results.get('metadatas', [[]])
        distances = results.get('distances', [[]])

        if documents and len(documents) > 0:
            doc_list  = documents[0] if isinstance(documents[0], list) else documents
            meta_list = metadatas[0] if metadatas and isinstance(metadatas[0], list) else (metadatas or [])
            dist_list = distances[0] if distances and isinstance(distances[0], list) else (distances or [])

            for i, doc in enumerate(doc_list):
                metadata   = meta_list[i] if i < len(meta_list) else {}
                distance   = dist_list[i]  if i < len(dist_list) else 1.0
                similarity = max(0.0, 1.0 - distance)

                retrieved_games.append({
                    'Platform':      metadata.get('Platform', ''),
                    'Name':          metadata.get('Name', ''),
                    'YearOfRelease': metadata.get('YearOfRelease', ''),
                    'Description':   metadata.get('Description', ''),
                    'Genre':         metadata.get('Genre', ''),
                    'Publisher':     metadata.get('Publisher', ''),
                    'Content':       doc,
                    'similarity':    round(similarity, 4),
                })

        return retrieved_games

    except Exception as e:
        print(f'retrieve_game error: {e}')
        return []


print('retrieve_game defined')

In [ ]:
# Test retrieve_game
test_results = retrieve_game.func('When was Pokemon Gold released?')
print(f'Documents retrieved: {len(test_results)}')
for r in test_results[:3]:
    print(f"  {r['Name']} ({r['YearOfRelease']})  similarity={r['similarity']}")

### Tool 2 — `evaluate_retrieval`

Acts as an **LLM judge**: given the user's question and the documents returned by `retrieve_game`, it decides whether those documents are sufficient (`useful=True`) or whether the agent should fall back to a web search (`useful=False`).

The structured output is parsed into an `EvaluationReport` instance using `PydanticOutputParser`.

In [ ]:
@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: Optional[List[Dict[str, Any]]] = None
) -> Dict[str, Any]:
    """
    Evaluates whether the retrieved documents are sufficient to answer the question.

    Workflow:
        1. Call retrieve_game(question) first.
        2. Then call evaluate_retrieval(question=..., retrieved_docs=<result from step 1>).

    Args:
        question:       The original user question.
        retrieved_docs: The list returned by retrieve_game().

    Returns:
        A dict with keys:
        - useful (bool): True if the docs are sufficient, False if web search is needed.
        - description (str): Explanation of the decision.
        - confidence (float): Numeric confidence score (0-1).
        - num_documents (int): Number of documents that were evaluated.
    """
    if not retrieved_docs:
        return {
            'useful':        False,
            'description':   'No documents were retrieved.',
            'confidence':    0.0,
            'num_documents': 0,
        }

    llm_judge = LLM(model='gpt-4o-mini', temperature=0.0, api_key=OPENAI_API_KEY)

    docs_text = '\n\n'.join([
        f"Game: {doc.get('Name', 'Unknown')} ({doc.get('YearOfRelease', 'Unknown')})\n"
        f"Platform: {doc.get('Platform', 'Unknown')}\n"
        f"Description: {doc.get('Description', 'No description')}"
        for doc in retrieved_docs
    ])

    evaluation_prompt = f"""Your task is to evaluate if the documents are enough to respond to the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Question: {question}

Retrieved Documents:
{docs_text}

Evaluate whether these documents contain sufficient information to answer the question.
Consider:
1. Do the documents directly address the question?
2. Is there enough detail to provide a complete answer?
3. Are there any gaps that would require additional sources?

Provide your evaluation with a clear explanation."""

    try:
        response   = llm_judge.invoke(input=evaluation_prompt, response_format=EvaluationReport)
        parser     = PydanticOutputParser(model_class=EvaluationReport)
        evaluation = parser.parse(response)

        similarities = [doc.get('similarity', 0.0) for doc in retrieved_docs if 'similarity' in doc]
        confidence   = 0.5
        if similarities:
            best       = max(similarities)
            avg        = sum(similarities) / len(similarities)
            confidence = best * 0.7 + avg * 0.3

        return {
            'useful':        evaluation.useful,
            'description':   evaluation.description,
            'confidence':    round(confidence, 3),
            'num_documents': len(retrieved_docs),
        }

    except Exception as e:
        return {
            'useful':        len(retrieved_docs) > 0,
            'description':   f'Parse error. {len(retrieved_docs)} docs evaluated. Error: {e}',
            'confidence':    0.3,
            'num_documents': len(retrieved_docs),
        }


print('evaluate_retrieval defined')

In [ ]:
# Test evaluate_retrieval
test_eval = evaluate_retrieval.func('When was Pokemon Gold released?', test_results)
print(f"useful      : {test_eval['useful']}")
print(f"confidence  : {test_eval['confidence']}")
print(f"description : {test_eval['description'][:200]}...")

### Tool 3 — `game_web_search`

A Tavily-powered web search that is called only when `evaluate_retrieval` returns `useful=False`. It returns a structured result including Tavily's synthesised answer and the individual web sources.

In [ ]:
@tool
def game_web_search(question: str) -> Dict[str, Any]:
    """
    Web search for video game topics using the Tavily API.
    Only call this tool after retrieve_game() and evaluate_retrieval() have confirmed
    that the internal database does not contain a sufficient answer.

    Args:
        question: A question about the video game industry.

    Returns:
        A dict with keys:
        - answer: Tavily's synthesised answer string.
        - results: list of {title, url, content, score}.
        - search_metadata: {query, result_count, source}.
    """
    if not TAVILY_API_KEY:
        raise ValueError('TAVILY_API_KEY is not set — check your .env file.')

    try:
        client        = TavilyClient(api_key=TAVILY_API_KEY)
        search_result = client.search(
            query=question,
            search_depth='advanced',
            include_answer=True,
            include_raw_content=False,
            include_images=False
        )

        return {
            'answer': search_result.get('answer', ''),
            'results': [
                {
                    'title':   r.get('title', ''),
                    'url':     r.get('url', ''),
                    'content': r.get('content', ''),
                    'score':   r.get('score', 0.0),
                }
                for r in search_result.get('results', [])
            ],
            'search_metadata': {
                'query':        question,
                'result_count': len(search_result.get('results', [])),
                'source':       'tavily_api',
            },
        }

    except Exception as e:
        raise RuntimeError(f'Web search failed: {e}')


print('game_web_search defined')

In [ ]:
# Test game_web_search — Palworld is NOT in the local DB
web_test = game_web_search.func('When was Palworld released and who developed it?')
print(f"Tavily answer  : {web_test['answer'][:300]}...")
print(f"Sources found  : {web_test['search_metadata']['result_count']}")

---

## Step 6 — Build the agent (Part 2)

The agent receives all three tools and a carefully written **system prompt**.

The system prompt uses numbered `STEP` instructions so the LLM cannot accidentally skip the retrieval-then-evaluation workflow. An `EXAMPLE` block acts as a few-shot demonstration of the expected call sequence.

In [ ]:
def create_udaplay_agent() -> Agent:
    """Create and return the UdaPlay agent with all tools attached."""

    system_instructions = """You are UdaPlay, an AI Research Agent specialised in answering questions about video games.

Your capabilities:
1. Search an internal knowledge base (vector database)
2. Evaluate whether retrieved information is sufficient
3. Search the web when the internal knowledge is insufficient
4. Generate clear, well-structured answers that cite sources

CRITICAL WORKFLOW — follow these steps in this exact order for EVERY question:

STEP 1: ALWAYS call retrieve_game(query) with the user's question.
STEP 2: ALWAYS call evaluate_retrieval(question=<user_question>, retrieved_docs=<result_from_step_1>).
        You MUST pass BOTH parameters.
STEP 3: Based on the evaluation:
        - If useful=True  → answer using the retrieved documents. Do NOT call game_web_search.
        - If useful=False → call game_web_search(question).
STEP 4: Generate the final answer and clearly state whether it came from the internal database or the web.

RULES:
- NEVER skip retrieve_game().
- NEVER call game_web_search() before completing STEP 1 and STEP 2.
- ALWAYS pass retrieved_docs to evaluate_retrieval().

EXAMPLE:
  User: "When was Pokemon Gold released?"
  Step 1 → retrieve_game("When was Pokemon Gold released?")
  Step 2 → evaluate_retrieval(question="When was Pokemon Gold released?", retrieved_docs=<step1_result>)
  Step 3 → useful=True → answer from retrieved documents.
  Step 4 → "Pokemon Gold was released in 1999 for the Game Boy Color. [Source: internal database]"
"""

    return Agent(
        model_name=   'gpt-4o-mini',
        instructions= system_instructions,
        tools=        [retrieve_game, evaluate_retrieval, game_web_search],
        temperature=  0.7
    )


agent = create_udaplay_agent()
print(f'Agent ready.  Model: {agent.model_name}  |  Tools: {len(agent.tools)}')

---

## Step 7 — Test queries

The helper function `run_query` sends a question to the agent and prints:
- the tool calls made (reasoning trace)
- the final answer
- metadata (source, confidence, document count)

**Expected behaviour:**
- Queries about games that ARE in the DB → `retrieve_game` + `evaluate_retrieval` → answer from internal DB
- Queries about games NOT in the DB (e.g. Palworld) → `useful=False` → `game_web_search` fallback
- Follow-up queries using the same `session_id` → agent uses conversation history

In [ ]:
def run_query(agent: Agent, query: str, session_id: str = 'default') -> None:
    """Run a single query and print the reasoning trace + final answer."""
    print(f'\nQuery: {query}')
    print('-' * 80)

    run         = agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state()
    messages    = final_state.get('messages', [])

    reasoning_steps         = []
    web_search_called       = False
    last_evaluation_useful  = True
    num_docs                = 0
    confidence              = None

    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                reasoning_steps.append(f'→ Called {tc.function.name}()')
                if tc.function.name == 'game_web_search':
                    web_search_called = True

        if hasattr(msg, 'name') and msg.name:
            try:
                import ast
                content_str = msg.content if isinstance(msg.content, str) else str(msg.content)
                try:
                    result = json.loads(content_str)
                except (json.JSONDecodeError, TypeError):
                    try:
                        result = ast.literal_eval(content_str)
                    except (ValueError, SyntaxError):
                        result = content_str

                if msg.name == 'retrieve_game' and isinstance(result, list):
                    num_docs = len(result)
                    if num_docs > 0:
                        best_sim = result[0].get('similarity') if isinstance(result[0], dict) else None
                        reasoning_steps.append(f'→ Retrieved {num_docs} documents')
                        if best_sim is not None:
                            reasoning_steps.append(f'   Best similarity: {best_sim:.3f}')
                    else:
                        reasoning_steps.append('→ No documents found')

                elif msg.name == 'evaluate_retrieval' and isinstance(result, dict):
                    useful = result.get('useful', False)
                    last_evaluation_useful = useful
                    confidence = result.get('confidence')  # always capture, even 0.0
                    conf_str = f'{confidence:.3f}' if confidence is not None else 'N/A'
                    reasoning_steps.append(f'→ Evaluation: useful={useful} | confidence={conf_str}')
                    if not useful:
                        reasoning_steps.append('→ Falling back to web search')

                elif msg.name == 'game_web_search' and isinstance(result, dict):
                    reasoning_steps.append('→ Web search completed')

            except Exception:
                pass

    source_type  = 'web_search' if (web_search_called or not last_evaluation_useful) else 'internal_database'
    final_answer = None
    for msg in reversed(messages):
        if hasattr(msg, 'content') and msg.content and not hasattr(msg, 'name'):
            final_answer = msg.content
            break

    print('\n[REASONING TRACE]')
    for step in reasoning_steps:
        print(f'  {step}')

    print(f'\n[ANSWER]\n{final_answer or "No answer generated."}')
    conf_display = f'{confidence:.3f}' if confidence is not None else 'N/A'
    print(f'\n[METADATA] Source: {source_type} | Confidence: {conf_display}', end='')
    if num_docs > 0:
        print(f' | Docs: {num_docs}', end='')
    print('\n' + '=' * 80)


print('run_query() defined')

In [ ]:
# Query 1 — game IS in the local DB
run_query(agent, 'When was Pokemon Gold and Silver released?')

In [ ]:
# Query 2 — game IS in the local DB
run_query(agent, 'Which one was the first 3D platformer Mario game?')

In [ ]:
# Query 3 — game IS in the local DB
run_query(agent, 'Who developed FIFA 21?')

In [ ]:
# Query 4 — game IS in the local DB
run_query(agent, 'What is the publisher name of Grand Theft Auto: San Andreas?')

In [ ]:
# Query 5 — game is NOT in the local DB → should trigger web search fallback
run_query(agent, 'When was Palworld released and who developed it?')

In [ ]:
# Query 6 — follow-up using the same session_id
# The agent should remember the Palworld answer from the previous turn
run_query(agent, 'What platform was it released on?', session_id='default')

---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `OPENAI_API_KEY not set` | `.env` missing or in wrong directory | Make sure `.env` is inside `starter/` |
| `401 Unauthorized` from OpenAI | Wrong endpoint in fallback branch | Re-apply **Fix 1** |
| `get() got unexpected keyword argument 'distances'` | `distances` left in `include` list | Re-apply **Fix 2** |
| `TypeError` when calling a tool | Optional params not filled in | Re-apply **Fix 3** |
| `401` from Tavily | Key invalid or quota exhausted | Check `TAVILY_API_KEY` in `.env` |
| Agent skips `retrieve_game` | System prompt too weak | Strengthen the `STEP` instructions |
| `games_store` is `None` | `setup_vector_database()` was not called | Re-run the setup cell |

---

## Optional enhancements

Once the basic agent works, here are ideas for extending it:

- **Long-term memory** — write useful web search results back into ChromaDB so future queries can use them.
- **More games** — add more JSON files to `games/` and re-run `setup_vector_database()`.
- **Structured final answer** — parse the agent's final response into a Pydantic model too.
- **Sentiment analysis tool** — a fourth tool that analyses player reviews.
- **Trending games detection** — use Tavily to surface recently released games.